In [1]:
import pandas as pd
from pathlib import Path

In [3]:
model = "dinov3_only_CLS"
checkpoint = "facebook_dinov3-vith16plus-pretrain-lvd1689m"
step_intervals = [1,3,5]

In [4]:
files_path = Path("data/files")
sim_path = files_path / f"similarities_{model}-{checkpoint}.csv"
sim_df = pd.read_csv(sim_path)
sim_df = sim_df[sim_df['metric'] == 'cosine'].reset_index(drop=True)
sim_df.head(5)

,metric,query_folder,query_image,query_mod,instr_folder,instr_image,instr_mod,score
0,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_1.jpg,NaN,0.850379
1,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_17.jpg,NaN,0.768050
2,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_3.jpg,NaN,0.759884
3,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_2.jpg,NaN,0.758960
4,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,NaN,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_15.jpg,NaN,0.741114


In [5]:
image_df = pd.read_csv("data/files/image_records.csv")
image_df['modification'] = image_df['modification'].fillna('')

In [6]:
sim_df['query_mod'] = sim_df['query_mod'].fillna('')

In [7]:
sim_df['instr_mod'] = sim_df['instr_mod'].fillna('')

In [8]:
sim_df.head(5)

,metric,query_folder,query_image,query_mod,instr_folder,instr_image,instr_mod,score
0,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_1.jpg,,0.850379
1,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_17.jpg,,0.768050
2,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_3.jpg,,0.759884
3,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_2.jpg,,0.758960
4,cosine,/home/drossi/experiment_1_dataset/45401-B_buil...,-90angle-clean-1.png,,/home/drossi/experiment_1_dataset/45401-B_buil...,instruction_step_15.jpg,,0.741114


In [9]:
grouped = sim_df.groupby(['query_folder','query_image','query_mod'])

In [10]:
step_lookup = image_df.set_index(['folder_path', 'image_name', 'modification'])['step_number']

best = sim_df.loc[grouped['score'].idxmax()].copy()
best['query_step'] = pd.MultiIndex.from_frame(
    best[['query_folder', 'query_image', 'query_mod']]).map(step_lookup)
best['instr_step'] = pd.MultiIndex.from_frame(
    best[['instr_folder', 'instr_image', 'instr_mod']]).map(step_lookup)
for interval in step_intervals:
    hf_interval = interval // 2
    best[f'{interval}_step_match'] = ((best['instr_step'] >= (best['query_step'] - hf_interval)) & (best['instr_step'] <= (best['query_step'] + hf_interval))).astype(float)    

best = best.rename(columns={'instr_folder': 'most_similar_instr_folder',
                     'instr_image': 'most_similar_instr_image',
                     'instr_mod': 'most_similar_instr_mod'})
step_match_path = files_path / f"step_match_{model}-{checkpoint}.csv"
best.to_csv(step_match_path, sep=',', index=False)

# Aggregating scores by category